# Milestone 2 — Hierarchical Parsing and Standardization

Notebook hoàn chỉnh chuyển corpus LaTeX thô (từ Milestone 1) thành dữ liệu có cấu trúc, chuẩn hoá và đã deduplicate, đúng ba bảng yêu cầu của đề: `elements`, `hierarchy` và `bibtex`.

**Toàn bộ thuật toán nằm trong mã nguồn của repo** — notebook chỉ cấu hình dữ liệu, gọi module và sinh thống kê/biểu đồ:

| Module | Nhiệm vụ | Đầu ra |
|---|---|---|
| `src/hierarchy.py` | gathering đa file, xây cây hierarchy, chuẩn hoá, dedup full-text | `elements.csv`, `hierarchy.csv` |
| `src/references.py` | parse `.bib` + `\bibitem`, dedup reference, union fields | `bibtex.csv` |

Pipeline: **gather → parse → (references) → dedup → 3 bảng + manifest + summary + biểu đồ**.


## Step 1 — Cấu hình đường dẫn

Tự tìm project root để chạy được cả local lẫn Colab. `CORPUS_ROOT` trỏ tới thư mục chứa các publication (không trỏ thẳng vào `v1`). Đổi corpus của thành viên khác chỉ cần đổi `CORPUS_ROOT`, không sửa thuật toán.


In [ ]:
from pathlib import Path
import csv
import json
import os
import re
import sys
from collections import Counter, defaultdict

def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'src' / 'hierarchy.py').is_file():
            return candidate
    raise FileNotFoundError('Không tìm thấy src/hierarchy.py. Hãy cd vào repository trước.')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---- mã nguồn của repo ----
from src.hierarchy import discover_publications, parse_corpus, parse_publication
from src.references import extract_bibtex

default_corpus = PROJECT_ROOT / '2512.18231-2512.23230' / 'corpus'
CORPUS_ROOT = Path(os.environ.get('CORPUS_ROOT', default_corpus))
OUTPUT_DIR = Path(os.environ.get('M2_OUTPUT_DIR', PROJECT_ROOT / 'output' / 'milestone2_notebook'))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('CORPUS_ROOT  =', CORPUS_ROOT)
print('OUTPUT_DIR   =', OUTPUT_DIR)
assert CORPUS_ROOT.is_dir(), f'Corpus không tồn tại: {CORPUS_ROOT}'


## Step 2 — Kiểm kê corpus

Phát hiện sớm lỗi mount/copy và xác nhận layout. Số version không có `.tex` vẫn được parser ghi là `empty` (không bỏ qua im lặng), đúng yêu cầu coverage.


In [ ]:
publications = discover_publications(CORPUS_ROOT)
version_dirs = [
    v for publication in publications
    for v in publication.iterdir()
    if v.is_dir() and re.fullmatch(r'v\d+', v.name, re.IGNORECASE)
]
versions_with_tex = sum(any(v.rglob('*.tex')) for v in version_dirs)
bib_files = sum(any(v.rglob('*.bib')) for v in version_dirs)

max_versions, most_versions = 0, None
for publication in publications:
    n = sum(1 for c in publication.iterdir() if c.is_dir() and re.fullmatch(r'v\d+', c.name, re.IGNORECASE))
    if n > max_versions:
        max_versions, most_versions = n, publication.name

inventory = {
    'publications': len(publications),
    'versions': len(version_dirs),
    'versions_with_tex': versions_with_tex,
    'versions_without_tex': len(version_dirs) - versions_with_tex,
    'versions_with_bib': bib_files,
    'max_versions': max_versions,
    'most_versions_publication': most_versions,
}
print(json.dumps(inventory, indent=2, ensure_ascii=False))
print('Ví dụ publication:', [p.name for p in publications[:5]])


## Step 3 — Demo một publication thật

Xem cây trước khi chạy batch. Parser tìm file `\documentclass`, lần theo `\input`/`\include` (bỏ file thừa), nhận diện cấp động `part`→`subparagraph`, giữ starred heading + Appendix + Acknowledgements, bỏ bibliography, tạo list/item, và giữ figure/table/equation thành leaf.


In [ ]:
PREFERRED_DEMO = '2512-20753'  # đổi thành folder của teammate nếu muốn
demo_dir = next((p for p in publications if p.name == PREFERRED_DEMO), publications[0])
demo = parse_publication(demo_dir)

print(f'Publication: {demo.arxiv_id}')
print(f'Versions: {len(demo.versions)} | distinct elements: {len(demo.elements)} | edges: {len(demo.hierarchy)}')
for result in demo.versions:
    print(f'  v{result.version}: status={result.status}, root={result.root_file}, '
          f'files={len(result.source_files)}, warnings={len(result.warnings)}')


In [ ]:
ok_version = next((v for v in demo.versions if v.status == 'ok'), demo.versions[0])
print(ok_version.root.pretty(max_content=100, max_depth=3))


## Step 4 — Kiểm tra các quy tắc bắt buộc

Biến yêu cầu đề bài thành điều kiện tái chạy được: bibliography không lọt vào hierarchy, list chỉ chứa `item`, block math chuẩn hoá về `\begin{equation}`, figure/table là leaf, và dedup full-text giữa các version.


In [ ]:
nodes = list(ok_version.root.walk())
kind_counts = Counter(node.kind for node in nodes)
all_content = '\n'.join(node.content for node in nodes).lower()

print('Node counts:', dict(kind_counts))
print('Starred headings:', sum(node.kind == 'heading' and node.numbered is False for node in nodes))
print('Appendix headings:', [node.content for node in nodes if node.kind == 'heading' and 'appendi' in node.content.lower()])
print('Acknowledgement headings:', [node.content for node in nodes if node.kind == 'heading' and 'acknow' in node.content.lower()])

assert '\\begin{thebibliography}' not in all_content
assert '\\bibliography{' not in all_content
assert all(child.kind == 'item' for node in nodes if node.kind == 'list' for child in node.children)
assert all(node.content.startswith('\\begin{equation}') for node in nodes if node.kind == 'block_formula')
assert all(len(node.children) == 0 for node in nodes if node.kind == 'figure')

if len(demo.versions) > 1:
    first, second = demo.versions[0], demo.versions[1]
    f = {n.content: n.element_id for n in first.root.walk() if n.kind == 'sentence'}
    s = {n.content: n.element_id for n in second.root.walk() if n.kind == 'sentence'}
    shared = set(f) & set(s)
    same_id = all(f[c] == s[c] for c in shared)
    print(f'Cross-version shared sentences: {len(shared)} | same element_id: {same_id}')
    assert same_id

print('Các invariant của hierarchy: PASS')


## Step 5 — Chạy batch (elements + hierarchy)

Mặc định chạy smoke 25 publication để sinh đủ ba bảng + biểu đồ (vài giây). Khi `versions_error=0`, đổi `BATCH_LIMIT=None` để chạy toàn bộ corpus (khoảng 20–40 phút).


In [ ]:
RUN_BATCH = True            # True: chạy batch để sinh đủ 3 bảng + biểu đồ; False: chỉ xem demo
BATCH_LIMIT = 25            # None = toàn bộ corpus (20–40 phút)
BIBTEX_LIMIT = BATCH_LIMIT  # giữ đồng bộ với batch (None = toàn bộ)

if RUN_BATCH:
    summary = parse_corpus(CORPUS_ROOT, OUTPUT_DIR, limit=BATCH_LIMIT)
    print(json.dumps(summary.as_dict(), indent=2, ensure_ascii=False))
else:
    print('Chưa chạy batch. Đặt RUN_BATCH = True sau khi demo một publication đã đúng.')


## Step 6 — Trích xuất & deduplicate BibTeX

`src.references.extract_bibtex` đọc `\bibitem` **chỉ từ compilation path** (không quét mọi `.tex`) và các file `.bib` được khai báo, rồi deduplicate cross-version (union fields, một `citation_key`). Kết quả ghi `bibtex.csv` + `bibtex_aliases.json` vào cùng `OUTPUT_DIR`.


In [ ]:
if RUN_BATCH:
    BIBTEX_STATS, bibtex_rows, bibtex_aliases = extract_bibtex(CORPUS_ROOT, OUTPUT_DIR, limit=BIBTEX_LIMIT)
    print(json.dumps(BIBTEX_STATS, indent=2, ensure_ascii=False))
else:
    BIBTEX_STATS, bibtex_rows, bibtex_aliases = {}, [], {}
    print('Chưa chạy BibTeX. Chạy Step 5 trước (RUN_BATCH=True).')


## Step 7 — Xác minh schema 3 bảng + foreign key + non-null

`elements` dùng chung giữa các version: cùng nội dung sau cleanup → cùng `element_id` (ví dụ `2512.20753#s17`). `hierarchy` lưu cạnh riêng theo version nên cùng một câu có thể gắn vào hai parent khác nhau ở v1/v2. Bước này đọc header từng CSV, kiểm tra `hierarchy.element_id` / `hierarchy.parent_id` đều nằm trong `elements.element_id`, và cột bắt buộc không null (streaming, không nạp toàn bộ RAM).


In [ ]:
expected_headers = {
    'elements.csv': ['element_id', 'arxiv_id', 'kind', 'content'],
    'hierarchy.csv': ['arxiv_id', 'version', 'element_id', 'parent_id'],
    'bibtex.csv': ['arxiv_id', 'citation_key', 'entry_type', 'title', 'authors', 'year', 'doi', 'journal', 'url'],
}

for filename, expected in expected_headers.items():
    path = OUTPUT_DIR / filename
    if not path.is_file():
        print(f'⚠ {filename} chưa tồn tại — bỏ qua.')
        continue
    with path.open(encoding='utf-8') as fh:
        reader = csv.DictReader(fh)
        header = reader.fieldnames
        sample = [next(reader) for _ in range(3)]
    ok = header == expected
    print(f'=== {filename} ===  header {"✓" if ok else "✗"}')
    print('  columns:', header)
    for row in sample:
        short = {k: (v[:60] + '…' if len(v) > 60 else v) for k, v in row.items()}
        print('  ', json.dumps(short, ensure_ascii=False))
    print()

elements_path = OUTPUT_DIR / 'elements.csv'
hierarchy_path = OUTPUT_DIR / 'hierarchy.csv'
if elements_path.is_file() and hierarchy_path.is_file():
    element_ids, null_element = set(), 0
    with elements_path.open(encoding='utf-8') as fh:
        for row in csv.DictReader(fh):
            element_ids.add(row['element_id'])
            if not row['element_id'] or not row['arxiv_id'] or not row['kind']:
                null_element += 1
    missing_child = missing_parent = edges = 0
    with hierarchy_path.open(encoding='utf-8') as fh:
        for row in csv.DictReader(fh):
            edges += 1
            if row['element_id'] not in element_ids:
                missing_child += 1
            if row['parent_id'] not in element_ids:
                missing_parent += 1
    fk_ok = missing_child == 0 and missing_parent == 0
    print(f'elements: {len(element_ids)} rows | hierarchy: {edges} edges')
    print(f'Foreign key: {"✓ OK" if fk_ok else f"✗ child={missing_child}, parent={missing_parent}"}')
    print(f'Non-null (elements): {"✓ OK" if null_element == 0 else f"✗ {null_element} rows"}')


## Step 8 — Biểu đồ & bảng tổng hợp cho report

Bốn biểu đồ được sinh ngay khi notebook chạy (không dùng ảnh tĩnh): version coverage, phân bố loại element, thời gian parse mỗi version, số node mỗi publication; kèm bảng tổng hợp benchmark.


In [ ]:
manifest_path = OUTPUT_DIR / 'manifest.jsonl'
summary_path = OUTPUT_DIR / 'summary.json'
elements_path = OUTPUT_DIR / 'elements.csv'
bibtex_path = OUTPUT_DIR / 'bibtex.csv'

manifest = []
if manifest_path.is_file():
    manifest = [json.loads(line) for line in manifest_path.read_text(encoding='utf-8').splitlines() if line.strip()]

if manifest:
    statuses = Counter(row['status'] for row in manifest)
    times = [row.get('elapsed_seconds', 0.0) for row in manifest]
    pub_nodes = defaultdict(int)
    for row in manifest:
        pub_nodes[row['arxiv_id']] += row.get('node_count', 0)

    kinds = Counter()
    if elements_path.is_file():
        with elements_path.open(encoding='utf-8') as fh:
            for row in csv.DictReader(fh):
                kinds[row['kind']] += 1

    try:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(2, 2, figsize=(13, 10))

        labels = ['ok', 'empty', 'error']
        vals = [statuses.get(l, 0) for l in labels]
        ax = axes[0][0]
        ax.bar(labels, vals, color=['#2ca02c', '#ffbf00', '#d62728'])
        ax.set(title='Version parse status', ylabel='Versions')
        ax.bar_label(ax.containers[0])

        ax = axes[0][1]
        top = kinds.most_common(7)
        ax.pie([v for _, v in top], labels=[k for k, _ in top], autopct='%1.1f%%', startangle=90)
        ax.set_title('Distribution of element types')

        ax = axes[1][0]
        ax.hist(times, bins=50, log=True)
        ax.set(title='Parse time per version (log scale)', xlabel='seconds')

        ax = axes[1][1]
        ax.boxplot(list(pub_nodes.values()), vert=True)
        ax.set(title='Nodes per publication', ylabel='node count')

        plt.tight_layout()
        plt.show()
    except ImportError:
        print('Không có matplotlib; số liệu text vẫn hợp lệ.')

    s = json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.is_file() else {}
    bib_count = sum(1 for _ in open(bibtex_path, encoding='utf-8')) - 1 if bibtex_path.is_file() else 0
    n_ver = max(s.get('versions', 1), 1)
    rows = [
        ('Metric', 'Value'),
        ('Publications processed', s.get('publications', '?')),
        ('Total versions', s.get('versions', '?')),
        ('Versions OK / empty / error', f"{s.get('versions_ok','?')} / {s.get('versions_empty','?')} / {s.get('versions_error','?')}"),
        ('Success rate', f"{100 * s.get('versions_ok', 0) / n_ver:.1f}%"),
        ('Elements (deduplicated)', s.get('elements', '?')),
        ('Hierarchy edges', s.get('hierarchy_edges', '?')),
        ('BibTeX entries (deduped)', bib_count),
        ('Total parse time (s)', f"{s.get('elapsed_seconds', 0):.1f}"),
        ('Avg time/publication (s)', f"{s.get('average_seconds_per_publication', 0):.3f}"),
        ('Peak RAM (MB)', f"{s.get('peak_ram_mb', 0):.1f}"),
        ('Output size (bytes)', s.get('output_bytes', '?')),
    ]
    print('\n'.join(f'| {k} | {v} |' for k, v in rows))
else:
    print('Chưa có manifest. Hãy chạy Step 5 trước (RUN_BATCH=True).')


## Step 9 — Phân tích chất lượng dữ liệu

Success rate (target > 85%), hiệu quả dedup BibTeX, phân tích lỗi/warning, và một sample tree của publication phức tạp.


In [ ]:
if manifest:
    total = len(manifest)
    ok = statuses.get('ok', 0)
    empty = statuses.get('empty', 0)
    error = statuses.get('error', 0)
    print(f'Success rate: {100 * ok / max(total, 1):.1f}%  (ok={ok}, empty={empty}, error={error} / {total})')

    raw = BIBTEX_STATS.get('raw_entries', 0)
    ded = BIBTEX_STATS.get('deduped_entries', 0)
    if raw:
        saving = 100 * (1 - ded / max(raw, 1))
        print(f'BibTeX dedup: {raw} raw -> {ded} dedup (tiết kiệm {saving:.1f}%), '
              f'{BIBTEX_STATS.get("aliases", 0)} aliases')

    errors = [m for m in manifest if m['status'] == 'error']
    if errors:
        err_types = Counter(w.split(':')[0] for m in errors for w in m.get('warnings', []))
        print(f'Error versions: {len(errors)}')
        print('Top error types:', dict(err_types.most_common(10)))
    else:
        print('Không có version lỗi.')

    warned = [m for m in manifest if m.get('warnings')]
    print(f'Versions with warnings: {len(warned)}')

    complex_name = next((m['publication_dir'] for m in manifest
                         if m['status'] == 'ok' and m.get('node_count', 0) > 150), None)
    if complex_name:
        pub_dir = next((p for p in publications if p.name == complex_name), None)
        if pub_dir:
            t = parse_publication(pub_dir)
            v = next((x for x in t.versions if x.status == 'ok'), t.versions[0])
            print(f'Sample tree ({t.arxiv_id}, v{v.version}):')
            print(v.root.pretty(max_content=80, max_depth=4))
else:
    print('Chưa có manifest. Hãy chạy Step 5 trước.')


## Step 10 — Checklist trước khi nộp

1. `python -m unittest discover -s tests -v` — unit tests pass (gồm `test_hierarchy.py` và `test_references.py`).
2. Demo ít nhất một bài multi-file, một bài có Appendix/Acknowledgements, một bài có list/formula/figure.
3. Smoke batch (25 publication mặc định): đọc toàn bộ `status=error` và `warnings`.
4. Full corpus: lưu năm file output nhỏ (`elements/hierarchy/bibtex.csv`, `manifest.jsonl`, `summary.json`); không commit raw corpus vào Git.
5. Ba bảng đúng schema + foreign key integrity + non-null (Step 7).
6. Bibliography bị loại khỏi hierarchy; references được xử lý riêng bởi `src/references.py`.
